In [1]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, gaussian_dist_time_varying, visualize_gaussian_samples, visualize_multi_task_gaussians, visualize_train_vs_test_gaussians, plot_disturbances_time, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, meta_train_nn_policy, maml_adapt, evaluate_policy_cost, plot_maml_diagnostics, plot_adapted_losses, plot_adaptation_comparison, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [2]:
# LINEARIZED QUADCOPTER DYNAMICS
A, B, C, D = quad_hover_linear_model()
A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]

In [3]:
# TRAINING TASKS
point_per_task = 50
task_means  = [-6.0, -4.0, -1.0, 2.0]
task_sigmas = [ 0.2,  0.4,  0.2, 0.3]


tasks_dx = []
tasks_all_disturbances = []
tasks_all_x0 = []

for mean, sigma in zip(task_means, task_sigmas):

    # time-varying disturbance on x
    dx = gaussian_dist_time_varying(mean, sigma, point_per_task)
    tasks_dx.append(dx)

    # nothing on y, gravity on z
    dy = torch.zeros_like(dx)
    dz = -cfg.G * torch.ones_like(dx)

    # combine to get full disturbance for this task, then append to all tasks
    dist = torch.cat([dx, dy, dz], dim=-1)      # shape [50, T, 3]
    tasks_all_disturbances.append(dist)         # shape [num_tasks, 50, T, 3]

    # initial states for all trajectories in this task
    x0 = torch.zeros(point_per_task, 12)
    tasks_all_x0.append(x0)

# just for baseline training
tasks_all_disturbances_joint = torch.cat(tasks_all_disturbances, dim=0)   # [num_tasks * point_per_task, T, 3]
tasks_all_x0_joint = torch.cat(tasks_all_x0, dim=0).float()               # [num_tasks * point_per_task, 12]


# TESTING POINTS
point_per_task = 50
nb_testing_points = 1 * point_per_task

task_target_mean = [6.0]
task_target_sigma = [0.3]
task_target = gaussian_dist_time_varying(task_target_mean[0], task_target_sigma[0], point_per_task, device="cpu")
tasks_target = [task_target]

dist_x_test = torch.cat([task_target], dim=0)                                       # x-axis disturbance (wind)                 
dist_y_test = torch.zeros_like(dist_x_test)                                         # y-axis disturbance (none)                  
dist_z_test = -cfg.G * torch.ones_like(dist_x_test)                                 # z-axis disturbance (gravity)
disturbances_test = torch.cat([dist_x_test, dist_y_test, dist_z_test], dim=-1)                        

# Initial state conditions for testing
x0_test_np = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]*nb_testing_points)
x0_test = torch.from_numpy(x0_test_np).float()


# VISUALIZE TRAINING/TESTING DISTURBANCES
visualize_multi_task_gaussians(tasks_dx, task_means, task_sigmas, "results_only_x/training_dist.png")
visualize_multi_task_gaussians(tasks_target, task_target_mean, task_target_sigma, "results_only_x/testing_dist.png", True)
visualize_train_vs_test_gaussians(tasks_dx, task_means, task_sigmas, tasks_target, task_target_mean, task_target_sigma, "results_only_x/both_training_testing_dist.png")

In [252]:
# JOINT TRAINING (BASELINE)
device = "cpu"
epochs = 4900
compute_mean_every = 100
lr = 4e-3

joint_policy, train_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=tasks_all_x0_joint, disturbances=tasks_all_disturbances_joint,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True, use_linearized=False
)

/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:217: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:218: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)


[ep 100/4900]:   train_loss=1.9671e+05   |    eval_loss=1.7275e+05   |   |x_T|_avg=123.918
[ep 200/4900]:   train_loss=1.6991e+05   |    eval_loss=1.7713e+05   |   |x_T|_avg=96.436
[ep 300/4900]:   train_loss=1.7678e+05   |    eval_loss=1.1014e+05   |   |x_T|_avg=97.545
[ep 400/4900]:   train_loss=8.2499e+04   |    eval_loss=9.5108e+04   |   |x_T|_avg=83.680
[ep 500/4900]:   train_loss=7.3604e+04   |    eval_loss=6.6214e+04   |   |x_T|_avg=83.295
[ep 600/4900]:   train_loss=6.5561e+04   |    eval_loss=9.6506e+04   |   |x_T|_avg=88.523
[ep 700/4900]:   train_loss=9.3728e+04   |    eval_loss=5.8357e+04   |   |x_T|_avg=35.082
[ep 800/4900]:   train_loss=4.5149e+04   |    eval_loss=4.4432e+04   |   |x_T|_avg=36.932
[ep 900/4900]:   train_loss=4.2880e+04   |    eval_loss=3.9978e+04   |   |x_T|_avg=62.168
[ep 1000/4900]:   train_loss=3.5760e+04   |    eval_loss=3.6854e+04   |   |x_T|_avg=53.586
[ep 1100/4900]:   train_loss=4.3386e+04   |    eval_loss=4.7659e+04   |   |x_T|_avg=75.019
[ep 120

In [ ]:
# PLOT LOSS JOINT TRAINING
plot_loss_evolution(train_means, epochs, compute_mean_every, filename="results_only_x/loss_joint_training.png")

In [6]:
# META TRAINING
device = "cpu"
epochs = 5000
compute_mean_every = 100
lr_outer_loop = 1e-3            # β
lr_inner_loop = 1e-3            # α
nb_samples_inner_training = 5
nb_samples_inner_testing = 5


meta_policy, logs = meta_train_nn_policy(
    A_d_nt = A_d, B_tilde_d_nt = B_tilde_d,
    init_points_list = tasks_all_x0,          
    disturbances_list = tasks_all_disturbances,
    lr_outer=lr_outer_loop,          
    lr_inner=lr_inner_loop,   
    meta_batch_size=None, 
    K_inner=nb_samples_inner_training,               
    K_outer=nb_samples_inner_testing,    
    epochs=epochs,
    device="cpu",
    verbose_every=compute_mean_every,
    use_linearized=False
)

[MAML] Meta-training on 4 tasks.


/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:389: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_i_prime, (x,))


[ep 100/5000] meta_loss = 2.8441e+06
[ep 200/5000] meta_loss = 1.0390e+06
[ep 300/5000] meta_loss = 5.4503e+05
[ep 400/5000] meta_loss = 2.3389e+05
[ep 500/5000] meta_loss = 1.3461e+05
[ep 600/5000] meta_loss = 1.2517e+05
[ep 700/5000] meta_loss = 1.5231e+05
[ep 800/5000] meta_loss = 7.0564e+04
[ep 900/5000] meta_loss = 4.3363e+04
[ep 1000/5000] meta_loss = 3.0064e+04
[ep 1100/5000] meta_loss = 2.5190e+04
[ep 1200/5000] meta_loss = 2.0758e+04
[ep 1300/5000] meta_loss = 2.7451e+04
[ep 1400/5000] meta_loss = 1.7011e+04
[ep 1500/5000] meta_loss = 8.6309e+03
[ep 1600/5000] meta_loss = 1.8841e+04
[ep 1700/5000] meta_loss = 5.2757e+03
[ep 1800/5000] meta_loss = 4.6118e+03
[ep 1900/5000] meta_loss = 3.7562e+03
[ep 2000/5000] meta_loss = 2.8897e+03
[ep 2100/5000] meta_loss = 2.4397e+03
[ep 2200/5000] meta_loss = 1.7695e+03
[ep 2300/5000] meta_loss = 1.4103e+03
[ep 2400/5000] meta_loss = 1.1385e+03
[ep 2500/5000] meta_loss = 9.3021e+02
[ep 2600/5000] meta_loss = 7.1812e+02
[ep 2700/5000] meta_l

In [7]:
plot_maml_diagnostics(logs, filename="results_only_x/loss_meta_training.png")

In [ ]:
# ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
torch.manual_seed(70)

adapted_policy_from_meta_learning, adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta = maml_adapt(
    meta_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=8e-2,
    nb_gradient_descent=6,
    K_samples=5,
    device="cpu",
    use_linearized=False
)
plot_adapted_losses(adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, filename="results_only_x/meta_adaptation.png")

/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:442: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:443: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)
/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:465: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead whi

[Adapt 1] inner_loss=1783575.7500, full_mean=1634071.5000
[Adapt 2] inner_loss=7108804.0000, full_mean=7267322.0000
[Adapt 3] inner_loss=64065.6992, full_mean=78818.7109
[Adapt 4] inner_loss=3897.6863, full_mean=3792.8538
[Adapt 5] inner_loss=73149.2891, full_mean=54355.2734
[Adapt 6] inner_loss=2345.7817, full_mean=2314.0811


In [310]:
# APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
torch.manual_seed(70)

adapted_policy_from_joint_learning, adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint = maml_adapt(
    joint_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=2e-3,
    nb_gradient_descent=6,
    K_samples=5,
    device="cpu",
    use_linearized=False
)
plot_adapted_losses(adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, filename="results_only_x/joint_adaptation.png")

[Adapt 1] inner_loss=273500.0000, full_mean=338525.8750
[Adapt 2] inner_loss=107241.1484, full_mean=141861.4219
[Adapt 3] inner_loss=27262.6523, full_mean=35749.0586
[Adapt 4] inner_loss=11297.8721, full_mean=13927.5430
[Adapt 5] inner_loss=6772.1274, full_mean=7626.4424
[Adapt 6] inner_loss=6024.2686, full_mean=6010.3770


In [ ]:
# COMPARE ADAPTED POLICIES ON TEST TASK
plot_adaptation_comparison(full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, "results_only_x/adaptation_comparison.png")

In [208]:
# TESTING ADAPTED POLICY FROM META-LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(meta_policy, adapted_policy_from_meta_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_meta, U_test_adapted_from_meta = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device),
    linearized=False
)
costs = traj_cost(X_test_adapted_from_meta, U_test_adapted_from_meta)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_meta, filename="results_only_x/testing_xyz_adapted_from_meta.png")
plot_angles_mean(X_test_adapted_from_meta, filename="results_only_x/testing_angles_adapted_from_meta.png")
plot_controls_grid_mean(U_test_adapted_from_meta, filename="results_only_x/testing_controls_adapted_from_meta.png")
animate_trajectories(X_test_adapted_from_meta, "results_only_x/testing_xyz_3d_adapted_from_meta.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_60516/731443896.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_60516/731443896.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_60516/731443896.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  retur

Adapted test cost = 1106.1947021484375
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [308]:
# TESTING ADAPTED POLICY FROM JOINT LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(joint_policy, adapted_policy_from_joint_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_joint, U_test_adapted_from_joint = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device),
    linearized=False
)
costs = traj_cost(X_test_adapted_from_joint, U_test_adapted_from_joint)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_joint, filename="results_only_x/testing_xyz_adapted_from_joint.png")
plot_angles_mean(X_test_adapted_from_joint, filename="results_only_x/testing_angles_adapted_from_joint.png")
plot_controls_grid_mean(U_test_adapted_from_joint, filename="results_only_x/testing_controls_adapted_from_joint.png")
animate_trajectories(X_test_adapted_from_joint, "results_only_x/testing_xyz_3d_adapted_from_joint.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_60516/3903944304.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_60516/3903944304.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_60516/3903944304.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  re

Adapted test cost = 34736.8984375
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200
